# معمل التجربة والتطبيق العملي: فوائد وحالات استخدام البحث الهجين
## Playground: Benefits and Enterprise Use Cases of Hybrid Search (`Lecture 43`)

مرحباً بك في المعمل العملي للمحاضرة 43 من مسار **RAG Mastery**.

---

### أهداف المعمل:
1. تجربة سيناريوهات الفشل الشائعة للمسترجع الكثيف بمفرده والمسترجع المتناثر بمفرده.
2. اختبار قدرة `EnsembleRetriever` على تغطية حالات الكلمات النادرة (`Rare Entities`) والمصطلحات التقنية.
3. تجربة الاستعلامات المركبة (`Mixed-Intent Queries`) التي تضم مترادفات ومصطلحات دقيقة معاً.
4. بناء دالة ذكية لضبط الأوزان ديناميكياً (`Dynamic Alpha Router`) بناءً على نمط الاستعلام.


In [ ]:
# 1. إعداد البيئة واستيراد المكتبات
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers.bm25 import BM25Retriever

try:
    from langchain.retrievers.ensemble import EnsembleRetriever
except ImportError:
    from langchain_classic.retrievers.ensemble import EnsembleRetriever

print("[INFO] المكتبات جاهزة للعمل في المعمل!")


In [ ]:
# 2. تجهيز مجموعة المستندات المتنوعة
corpus = [
    Document(page_content="API usage: call client.chat.completions.create(stream=True) for streaming.", metadata={"domain": "api"}),
    Document(page_content="Clinical diagnosis of osteoporosis requires bone mineral density T-score <= -2.5.", metadata={"domain": "medical"}),
    Document(page_content="Affordable ANC wireless earbuds with active noise cancellation.", metadata={"domain": "ecommerce"}),
    Document(page_content="Architectural blueprint for building autonomous reasoning agent systems.", metadata={"domain": "ai"}),
    Document(page_content="Compliance clause 402-B mandates encrypted storage for financial records.", metadata={"domain": "legal"}),
    Document(page_content="European castles, gothic cathedrals, and historic medieval towers.", metadata={"domain": "general"})
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(corpus, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

sparse_retriever = BM25Retriever.from_documents(corpus)
sparse_retriever.k = 3

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
)

print(f"[INFO] تم تهيئة الفهارس بنجاح لـ {len(corpus)} مستندات.")


---
### مساحة التجربة والتطبيق العملي 1: اختبار الألفاظ النادرة وغير المألوفة (Out-Of-Vocabulary)

**المطلوب:**
قم بالبحث عن كود تقني نادر أو رمز خطأ برمجي مثل: `"402-B"` أو `"stream=True"`.
قارن بين ما يسترجعه المسترجع الكثيف وما يسترجعه المسترجع المتناثر والمسترجع الهجين.


In [ ]:
# مساحة التجربة والتطبيق العملي 1
query_code = "402-B"

# اختبر المسترجعات الثلاثة:
# dense_res = dense_retriever.invoke(query_code)
# sparse_res = sparse_retriever.invoke(query_code)
# hybrid_res = hybrid_retriever.invoke(query_code)

# اطبع النتائج وقارن ترتيب المستند القانوني المطلوب


---
### مساحة التجربة والتطبيق العملي 2: اختبار البحث الدلالي بدون كلمات مطابقة

**المطلوب:**
ابحث عن معنى وثيقة الذكاء الاصطناعي ولكن باستخدام كلمات مختلفة تماماً (مثلاً: `"constructing smart decision-making software bots"`).
لاحظ كيف يفشل `BM25` في العثور على أي تطابق، بينما ينجح المسترجع الكثيف والهجين في التقاطها.


In [ ]:
# مساحة التجربة والتطبيق العملي 2
query_semantic_only = "constructing smart decision-making software bots"

# جرب invoke وقارن المخرجات


---
### مساحة التجربة والتطبيق العملي 3: بناء موجه أوزان ديناميكي (Dynamic Weight Router)

**المطلوب:**
اكتب دالة بلغة بايثون تفحص استعلام المستخدم:
- إذا كان الاستعلام يحتوي على رموز خاصة (`=`, `_`, `-`) أو أرقام أو كلمات قصيرة بحروف كبيرة (مثل `ANC`, `API`)، تقوم الدالة بإنشاء `EnsembleRetriever` بوزن `[0.2, 0.8]` (تغليب المتناثر).
- وإذا كان الاستعلام جملة سردية عامة، تجعل الوزن `[0.8, 0.2]` (تغليب الكثيف).


In [ ]:
# مساحة التجربة والتطبيق العملي 3
# اكتب كود دالة الـ Router الذكية هنا
def get_adaptive_hybrid_retriever(query: str):
    # قم بفحص النص وتحديد الأوزان المناسبة
    pass
